> **Disclaimer.** This project is educational and is not investment advice. Backtest results are historical and do not predict future results.

# Book of trades: one configuration, run through the Python model

Pick a configuration, run the backtest on WIN (or WDO), and print the **last N trades** one by one, to compare with the ProfitPro backtest of `ntsl/noise_area_win.ntsl` (`SendOrders(true)`, `Contracts(1)`). The Python model enters and leaves at the **close of the decision bar** (a named exception to the next-open rule, brief section 6); the table also shows the **next bar's open** as the alternative fill, so either convention can be matched to what ProfitPro reports.

**Read this first**
- **The final test period has been read** (once, Nov 17, 2026; `winfut_final_test.ipynb`). Nothing here may be used to choose or tune anything. `INCLUDE_FINAL` only decides whether the trades after 2025-01-01 are shown; the last trades of the whole history are in that period.
- **To compare with ProfitPro**: same instrument and chart (1 minute), `Contracts` 1 there and `SIZING = "fixed"` with 1 contract here, the same costs (here: the B3 fee plus `SLIPPAGE_TICKS` ticks a side), and `SessionEnd` set to the real last close of the season (17:55 in the short-session seasons).
- The comparison period (3 to 16 November 2026) comes from a separate file in `samples/comparison/`, so the project loader keeps its one-file-per-symbol rule and its guard on the final period. The table shows trades, never candles. Outputs are stripped on commit (the default for notebooks).

In [ ]:
import os
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from b3core.intraday_momentum import performance, run_intraday_momentum, trade_list
from b3core.intraday_search import INSTRUMENTS, make_config
from b3core.winfut import load_bars

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 40
pd.options.display.max_rows = 200

## 1. Parameters

`PRESET`: `"paper"` is the paper's rule (14 days, bands x1.0, every 30 minutes from 09:00, held to the close, VWAP + band stop on the half-hour grid); `"WIN optimized"` is the best trial by WIN of the 50,000-trial search (20 days, x1.5, every 60 minutes from 12:00, exit 18:20, band stop checked on every bar). Anything else: edit `PARAMS` in the next cell.

`SIZING = "fixed"` trades `FIXED_CONTRACTS` a day (1 to compare with ProfitPro); `"vol_target"` is the paper's volatility target on a R$ 100,000 account.

In [ ]:
PRESET = os.environ.get("PRESET", "WIN optimized")  # "paper" or "WIN optimized"
INSTRUMENT = os.environ.get("INSTRUMENT", "WIN")  # "WIN" or "WDO"
SIZING = "fixed"  # "fixed" or "vol_target"
FIXED_CONTRACTS = 1
SLIPPAGE_TICKS = 1.0  # per side, on top of the B3 fee (the search used 1)
LAST_N = 50  # trades to print
INCLUDE_FINAL = True  # show trades after 2025-01-01 (display only: the final period has been read)
INCLUDE_NOVEMBER = True  # add the 3-16 Nov 2026 file in samples/comparison (kept out of the project loader), for the ProfitPro comparison
COMPARE_FROM = "2026-11-03"  # first day of the comparison period

PRESETS = {
    "paper": {"lookback": 14, "band_mult": 1.0, "trade_freq": 30, "first_decision": 540, "exit_minute": 1440,
              "stop": "vwap_band", "stop_check": "grid", "sides": "both"},
    "WIN optimized": {"lookback": 20, "band_mult": 1.5, "trade_freq": 60, "first_decision": 720, "exit_minute": 1100,
                      "stop": "current_band", "stop_check": "every_bar", "sides": "both"},
}
PARAMS = PRESETS[PRESET]
cfg = make_config(PARAMS, INSTRUMENT, slippage_ticks=SLIPPAGE_TICKS)
cfg = replace(cfg, fixed_contracts=FIXED_CONTRACTS if SIZING == "fixed" else None)
inst = INSTRUMENTS[INSTRUMENT]
print(f"{PRESET} on {INSTRUMENT}: {PARAMS}")
print(f"{SIZING} sizing" + (f", {FIXED_CONTRACTS} contract(s)" if SIZING == "fixed" else "")
      + f"; cost {cfg.fee_per_side:.2f} R$ a side (fee {inst.fee_per_side:.2f} + {SLIPPAGE_TICKS:g} tick x {inst.tick_reais:.2f}); point value {cfg.point_value:g}")

## 2. The run

In [ ]:
dev = load_bars("../samples", symbol=inst.symbol)
parts = [dev]
if INCLUDE_FINAL:
    parts.append(load_bars("../samples", symbol=inst.symbol, final=True))
if INCLUDE_NOVEMBER and INCLUDE_FINAL:
    nov = load_bars("../samples/comparison", symbol=inst.symbol, final=True)  # a separate folder: the loader takes one file per symbol
    assert nov.index.min() > parts[-1].index.max(), "the November file overlaps the history"
    parts.append(nov)
bars = pd.concat(parts)
result = run_intraday_momentum(bars, cfg)
trades = trade_list(bars, cfg, result)
print(f"bars {bars.index[0]:%Y-%m-%d} to {bars.index[-1]:%Y-%m-%d}: {len(trades):,} trades over {trades['day'].nunique():,} days")
whole = performance(result["ret"])
print(f"whole run: net {trades['net'].sum():,.0f} R$ on {FIXED_CONTRACTS if SIZING == 'fixed' else 'volatility-targeted'} contracts, "
      f"Sharpe {whole['sharpe']:.2f}, hit ratio of days {whole['hit_ratio']:.1%}")

## 3. The trades

Each row is one trade (a run of the same position inside a day; a reversal is two trades at the same price). `entry_time` is the close of the decision bar and `exit_time` the close of the last bar held. `points` use the Python fills (decision close). `next-open` columns repeat the trade with the **open of the bar after** the entry and the exit (the exit has no next bar when the day ends, so those are empty), which is the other way ProfitPro may fill a market order.

In [ ]:
def add_next_open(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    opens = bars["open"]
    out["entry_next_open"] = opens.reindex(pd.DatetimeIndex(out["entry_time"])).to_numpy()
    out["exit_next_open"] = opens.reindex(pd.DatetimeIndex(out["exit_time"])).to_numpy()
    out["points_next_open"] = out["side"] * (out["exit_next_open"] - out["entry_next_open"])
    out["minutes"] = ((out["exit_time"] - out["entry_time"]).dt.total_seconds() / 60).astype(int)
    out["cum_net"] = out["net"].cumsum()
    return out


book = add_next_open(trades)
shown = book.tail(LAST_N).copy()
print(f"last {len(shown)} trades: {shown['day'].iloc[0]:%Y-%m-%d} to {shown['day'].iloc[-1]:%Y-%m-%d}")
summary = pd.Series({
    "trades": len(shown), "long / short": f"{int((shown['side'] == 1).sum())} / {int((shown['side'] == -1).sum())}",
    "winners %": (shown["points"] > 0).mean() * 100, "points, total": shown["points"].sum(),
    "points, mean": shown["points"].mean(), "gross R$": shown["gross"].sum(), "cost R$": shown["cost"].sum(),
    "net R$": shown["net"].sum(), "best net R$": shown["net"].max(), "worst net R$": shown["net"].min(),
    "mean minutes held": shown["minutes"].mean(),
}, name=f"last {LAST_N}")
display(summary.to_frame())

In [ ]:
table = shown.assign(
    day=shown["day"].dt.strftime("%Y-%m-%d"), side=shown["side"].map({1: "long", -1: "short"}),
    entry=shown["entry_time"].dt.strftime("%H:%M"), exit=shown["exit_time"].dt.strftime("%H:%M"),
)[["day", "side", "entry", "exit", "entry_price", "exit_price", "points", "entry_next_open", "exit_next_open", "points_next_open",
   "contracts", "gross", "cost", "net", "cum_net"]]
table.index = range(len(book) - len(shown) + 1, len(book) + 1)
table.index.name = "trade #"
display(table)

## 3b. The comparison period, every trade

Every trade of the run from `COMPARE_FROM` on (the November 2026 file), to put next to the ProfitPro trade report. The strategy runs over the whole history, so the bands and the lookback are warm when November starts. **The last day, 2026-11-16, is incomplete in the file** (the export stops at 17:30): the Python model closes a trade at the last bar it has, which ProfitPro, with a full day, would not.

In [ ]:
cmp_ = book[book["day"] >= pd.Timestamp(COMPARE_FROM)].copy()
print(f"{len(cmp_)} trades from {COMPARE_FROM}: " + (f"{cmp_['day'].iloc[0]:%Y-%m-%d} to {cmp_['day'].iloc[-1]:%Y-%m-%d}" if len(cmp_) else "none"))
cmp_table = cmp_.assign(
    day=cmp_["day"].dt.strftime("%Y-%m-%d"), side=cmp_["side"].map({1: "long", -1: "short"}),
    entry=cmp_["entry_time"].dt.strftime("%H:%M"), exit=cmp_["exit_time"].dt.strftime("%H:%M"),
)[["day", "side", "entry", "exit", "entry_price", "exit_price", "points", "entry_next_open", "exit_next_open", "points_next_open", "contracts", "gross", "cost", "net"]]
cmp_table.index = range(1, len(cmp_table) + 1)
cmp_table.index.name = "#"
display(cmp_table)
per_day = cmp_.groupby("day").agg(trades=("net", "size"), points=("points", "sum"), net=("net", "sum"))
per_day["cum_net"] = per_day["net"].cumsum()
display(per_day.assign(day_label=per_day.index.strftime("%a")).set_index("day_label", append=True))
print(f"total: {len(cmp_)} trades, {cmp_['points'].sum():,.0f} points, net {cmp_['net'].sum():,.0f} R$; with the next-open fills {cmp_['points_next_open'].sum():,.0f} points")

## 4. Net result of each of the trades shown

Net of the B3 fee and the slippage, in reais, on the contracts above.

In [ ]:
fig = go.Figure(go.Bar(x=list(table.index), y=shown["net"].to_numpy(), marker_color=np.where(shown["net"].to_numpy() >= 0, "seagreen", "indianred")))
fig.update_layout(title=f"Net R$ per trade, last {len(shown)} trades ({PRESET}, {INSTRUMENT})", xaxis_title="trade #", yaxis_title="R$", height=320, margin=dict(t=60, b=30))
fig.show()

## 5. The whole book, to a file

The full list of trades of the run is written as a CSV (derived from the vendor data, so it is git-ignored) to open next to the ProfitPro trade report.

In [ ]:
OUT = Path("../samples/optimization")
OUT.mkdir(parents=True, exist_ok=True)
slug = PRESET.lower().replace(" ", "_")
path = OUT / f"trade_book_{INSTRUMENT}_{slug}.csv"
book.drop(columns=["cum_net"]).to_csv(path, index=False, float_format="%.4f")
cmp_path = OUT / f"trade_book_{INSTRUMENT}_{slug}_from_{COMPARE_FROM}.csv"
cmp_.drop(columns=["cum_net"]).to_csv(cmp_path, index=False, float_format="%.4f")
print(f"{len(book):,} trades written to {path}; the comparison period ({len(cmp_)} trades) to {cmp_path}")